In [ ]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from scipy.ndimage import gaussian_filter
import matplotlib.cm as cm


session_folder = r"C:\Users\ebonhomm\Desktop\20260515_164734"

sigma = 18
alpha_max = 0.70
colormap_name = "jet"

output_folder = os.path.join(session_folder, "overlays")
os.makedirs(output_folder, exist_ok=True)


data_csv = glob.glob(os.path.join(session_folder, "aoi_heatmap_data*.csv"))[0]
metadata_csv = glob.glob(os.path.join(session_folder, "aoi_metadata*.csv"))[0]
capture_files = glob.glob(os.path.join(session_folder, "aoi_capture_*.png"))

df = pd.read_csv(data_csv)
metadata = pd.read_csv(metadata_csv)

df = df[df["valid"] == 1]
df = df[df["yeux_fermes"] == 0]
df = df[(df["u"] >= 0) & (df["u"] <= 1)]
df = df[(df["v"] >= 0) & (df["v"] <= 1)]


def find_capture(aoi_id):
    for path in capture_files:
        if os.path.basename(path).startswith(f"aoi_capture_{aoi_id}_"):
            return path
    return None


def order_points(pts):
    pts = np.array(pts, dtype=np.float32)

    s = pts.sum(axis=1)
    diff = np.diff(pts, axis=1).reshape(-1)

    top_left = pts[np.argmin(s)]
    bottom_right = pts[np.argmax(s)]
    top_right = pts[np.argmin(diff)]
    bottom_left = pts[np.argmax(diff)]

    return np.array([bottom_left, bottom_right, top_right, top_left], dtype=np.float32)


def detect_aoi_quad_from_capture(image_rgba):
    img = np.array(image_rgba.convert("RGBA"))
    rgb = img[:, :, :3]
    alpha = img[:, :, 3]

    # AOI claire sur fond noir
    mask = (
        (alpha > 0) &
        (rgb[:, :, 0] > 40) &
        (rgb[:, :, 1] > 40) &
        (rgb[:, :, 2] > 40)
    ).astype(np.uint8) * 255

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        raise RuntimeError("Impossible de détecter l'AOI dans la capture.")

    contour = max(contours, key=cv2.contourArea)

    epsilon = 0.02 * cv2.arcLength(contour, True)
    approx = cv2.approxPolyDP(contour, epsilon, True)

    if len(approx) >= 4:
        pts = approx.reshape(-1, 2)

        if len(pts) > 4:
            rect = cv2.minAreaRect(contour)
            pts = cv2.boxPoints(rect)

        return order_points(pts)

    rect = cv2.minAreaRect(contour)
    pts = cv2.boxPoints(rect)

    return order_points(pts)


def make_rect_heatmap_texture(u, v, tex_w=1024, tex_h=1024):
    heatmap, _, _ = np.histogram2d(
        u,
        v,
        bins=[tex_w, tex_h],
        range=[[0, 1], [0, 1]]
    )

    heatmap = gaussian_filter(heatmap, sigma=sigma)

    if heatmap.max() > 0:
        heatmap = heatmap / heatmap.max()

    heatmap_img = np.flipud(heatmap.T)

    rgba = cm.get_cmap(colormap_name)(heatmap_img)
    rgba[:, :, 3] = heatmap_img * alpha_max

    return (rgba * 255).astype(np.uint8)


def warp_heatmap_to_aoi(heatmap_rgba, quad_points, output_size):
    width, height = output_size

    src_h, src_w = heatmap_rgba.shape[:2]

    src = np.array([
        [0, src_h - 1],          # bottom left
        [src_w - 1, src_h - 1],  # bottom right
        [src_w - 1, 0],          # top right
        [0, 0],                  # top left
    ], dtype=np.float32)

    dst = quad_points.astype(np.float32)

    matrix = cv2.getPerspectiveTransform(src, dst)

    warped = cv2.warpPerspective(
        heatmap_rgba,
        matrix,
        (width, height),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(0, 0, 0, 0)
    )

    return Image.fromarray(warped, mode="RGBA")


for _, row in metadata.iterrows():
    aoi_id = str(row["aoi_id"])

    df_aoi = df[df["aoi_id"] == aoi_id]

    if df_aoi.empty:
        print(f"Aucune donnée pour {aoi_id}")
        continue

    capture_path = find_capture(aoi_id)

    if capture_path is None:
        print(f"Capture introuvable pour {aoi_id}")
        continue

    background = Image.open(capture_path).convert("RGBA")
    width, height = background.size

    quad = detect_aoi_quad_from_capture(background)

    rect_heatmap = make_rect_heatmap_texture(
        df_aoi["u"].values,
        df_aoi["v"].values,
        tex_w=1024,
        tex_h=1024
    )

    warped_heatmap = warp_heatmap_to_aoi(
        rect_heatmap,
        quad,
        output_size=(width, height)
    )

    overlay = Image.alpha_composite(background, warped_heatmap)

    output_path = os.path.join(output_folder, f"overlay_heatmap_{aoi_id}.png")
    overlay.save(output_path)

    print(f"{aoi_id} -> {output_path}")